In [ ]:
"""
H3a: Per-SV Gradient Utilization -- Muon vs Normalized SGD
============================================================

MOTIVATION (from H3 surprise):
  Muon beats normalized SGD by 19x (linear) and 10x (ReLU) at optimal LR.
  The critical difference: scale normalization PRESERVES gradient anisotropy
  (keeps internal SV ratios), while the polar factor EQUALIZES them (all -> 1).
  This is qualitatively different.

QUESTION: How much gradient information does each optimizer USE per singular
  direction? If G = U diag(sigma) V^T, SGD uses sigma_i (favors large SVs),
  normalized SGD uses sigma_i / ||sigma|| (same ratios, just scaled), and
  Muon uses 1 for all i (equal contribution from every SV direction).

PROTOCOL:
  At each training step, decompose gradient G = U * diag(sigma) * V^T.
  Compute for each optimizer's actual update Delta_W:
    - Project Delta_W onto each left/right singular direction pair.
    - Measure "utilization_i" = |<Delta_W, u_i v_i^T>| for each SV direction.
    - Compute utilization entropy: H = -sum(p_i log p_i) where p_i = util_i / sum(util).
  Higher entropy = more democratic use of ALL gradient directions.

KEY TESTS:
  T1: Muon has higher utilization entropy than normalized SGD (>0.5 bits more).
  T2: Muon's utilization is nearly uniform (H > 0.9 * H_max).
  T3: SGD's utilization is dominated by top SV (top-1 fraction > 0.5).

Setup: 4-layer, 32x32, track utilization over 500 steps, 5 seeds.
"""

print("=" * 100)
print("H3a: PER-SINGULAR-VALUE GRADIENT UTILIZATION ENTROPY")
print("Experiment: Do optimizers use ALL gradient directions, or only the dominant ones?")
print("=" * 100)
print()
print("THEORETICAL BACKGROUND")
print("-" * 60)
print("Any gradient G in R^{m x n} has an SVD: G = U diag(sigma) V^T.")
print("Each rank-1 component u_i v_i^T defines an independent 'direction'")
print("in weight space. The question is: when an optimizer computes its")
print("update Delta_W, how much of that update aligns with each direction?")
print()
print("Define utilization_i = |<Delta_W, u_i v_i^T>| = |u_i^T Delta_W v_i|.")
print("Normalize: p_i = utilization_i / sum(utilization_j).")
print("Compute Shannon entropy: H = -sum(p_i log2(p_i)).")
print()
print("PREDICTIONS:")
print("  SGD:          Delta_W ~ G = U diag(sigma) V^T")
print("                => utilization_i ~ sigma_i (dominated by top SVs)")
print("                => LOW entropy (anisotropic utilization)")
print()
print("  Normalized SGD: Delta_W ~ G / ||G||_F = U diag(sigma/||sigma||) V^T")
print("                => utilization_i ~ sigma_i / ||sigma|| (SAME ratios)")
print("                => LOW entropy (still anisotropic, just rescaled)")
print()
print("  Muon:         Delta_W ~ U V^T (polar factor, all sigma_i -> 1)")
print("                => utilization_i ~ 1 for all i (uniform)")
print("                => HIGH entropy (maximally democratic)")
print()
print("This is the key structural insight: Muon does not just rescale the")
print("gradient -- it REDISTRIBUTES information across ALL singular directions.")
print("=" * 100)

In [ ]:
import numpy as np
import os

print("Imports loaded: numpy, os")
print(f"  numpy version: {np.__version__}")

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
print(f"Script directory: {SCRIPT_DIR}")

# Experimental Configuration

**Network architecture:** A deep linear network with `NUM_LAYERS` square layers of dimension `DIM x DIM`. This is a controlled setting where gradients have rich singular value structure but no nonlinearity-induced complications.

**Key parameters:**
- `DIM = 32`: Each weight matrix is 32x32, so gradients have 32 singular values. Maximum utilization entropy is log2(32) = 5.0 bits.
- `NUM_STEPS = 500`: Enough to observe steady-state utilization patterns, not just transient initialization effects.
- `MOMENTUM = 0.9`: Standard momentum, which introduces temporal mixing of gradient SVD structure across steps.
- `NS_ITERS = 5`: Newton-Schulz iterations for Muon's orthogonalization. 5 iterations gives high-precision convergence to the polar factor for well-conditioned inputs.
- `NUM_SEEDS = 5`: Multiple seeds for statistical robustness.

**Why these choices matter:** The 32x32 dimension is large enough that uniform vs. concentrated utilization is clearly distinguishable (H_max = 5.0 bits, so even a 0.5-bit difference is a 10% effect), but small enough for exact SVD at every step.

In [ ]:
DIM = 32
NUM_LAYERS = 4
NUM_STEPS = 500
MOMENTUM = 0.9
NS_ITERS = 5
NUM_SEEDS = 5
BATCH_SIZE = 64

H_max = np.log2(DIM)
print("EXPERIMENTAL CONFIGURATION")
print("-" * 50)
print(f"  Weight matrix dimension:    {DIM} x {DIM}")
print(f"  Number of layers:           {NUM_LAYERS}")
print(f"  Training steps per run:     {NUM_STEPS}")
print(f"  Momentum coefficient:       {MOMENTUM}")
print(f"  Newton-Schulz iterations:   {NS_ITERS}")
print(f"  Number of random seeds:     {NUM_SEEDS}")
print(f"  Batch size:                 {BATCH_SIZE}")
print()
print(f"  Maximum utilization entropy (log2({DIM})): {H_max:.4f} bits")
print(f"  Threshold for T1 (entropy gap):            0.5 bits")
print(f"  Threshold for T2 (near-uniform):           {0.9 * H_max:.4f} bits (0.9 * H_max)")
print(f"  Threshold for T3 (SGD top-1 dominance):    0.5 (fraction)")
print()
print(f"  Total measurements per method: {NUM_STEPS} steps x {NUM_LAYERS} layers x {NUM_SEEDS} seeds = {NUM_STEPS * NUM_LAYERS * NUM_SEEDS}")
print(f"  This gives {NUM_STEPS * NUM_LAYERS * NUM_SEEDS} utilization entropy samples per optimizer.")

# Core Functions: Newton-Schulz Orthogonalization and Network Operations

## Newton-Schulz Iteration for Polar Factor Extraction

The **polar decomposition** of a matrix M = U_p H decomposes M into an orthogonal factor U_p and a symmetric positive semi-definite factor H. The Newton-Schulz iteration computes U_p iteratively:

$$X_{k+1} = \frac{3}{2} X_k - \frac{1}{2} X_k (X_k^T X_k)$$

Starting from X_0 = M / ||M||_F, this converges cubically to the polar factor U_p.

**Why this matters for utilization entropy:** The polar factor U_p satisfies U_p = U V^T where M = U diag(sigma) V^T is the SVD. This means **all singular values are replaced by 1**, which is exactly why we predict Muon achieves uniform utilization across SV directions.

In [ ]:
def newton_schulz(M, n_iters=NS_ITERS):
    norm = np.linalg.norm(M, ord='fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

# Verify on a test matrix that NS produces near-orthogonal output
_test_M = np.random.randn(DIM, DIM)
_test_Q = newton_schulz(_test_M)
_test_QtQ = _test_Q.T @ _test_Q
_ortho_error = np.linalg.norm(_test_QtQ - np.eye(DIM), 'fro')
print(f"Newton-Schulz verification (random {DIM}x{DIM} matrix, {NS_ITERS} iterations):")
print(f"  ||Q^T Q - I||_F = {_ortho_error:.2e}  (should be < 1e-6 for good convergence)")
_test_U, _test_s, _test_Vt = np.linalg.svd(_test_Q, full_matrices=False)
print(f"  Singular values of Q: min={_test_s.min():.6f}, max={_test_s.max():.6f}")
print(f"  (All should be ~1.0 for a proper orthogonal matrix)")
del _test_M, _test_Q, _test_QtQ, _ortho_error, _test_U, _test_s, _test_Vt

## Network Initialization, Forward Pass, and Gradient Computation

**Weight initialization:** Each layer is initialized as I + 0.1 * noise, i.e., a near-identity perturbation. This ensures:
1. The initial network is close to a linear map (product of near-identities ~ identity)
2. Initial gradients have non-trivial SVD structure (not purely isotropic)
3. Training is numerically stable from the start

**Forward pass:** y = W_L ... W_2 W_1 x (deep linear composition)

**Gradient computation:** Standard backpropagation through the linear chain. For layer l: grad_l = delta_l @ activations_l^T, where delta propagates backward through the transpose weights.

In [ ]:
def init_weights(seed):
    rng = np.random.RandomState(seed)
    return [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(NUM_LAYERS)]

# Quick diagnostic: what does initial gradient structure look like?
_test_weights = init_weights(42)
print(f"Initial weight structure (seed=42):")
for _l, _W in enumerate(_test_weights):
    _s = np.linalg.svd(_W, compute_uv=False)
    print(f"  Layer {_l}: sigma_range=[{_s.min():.4f}, {_s.max():.4f}], "
          f"condition_number={_s.max()/_s.min():.4f}")
print(f"  (Near-identity: condition numbers close to 1 indicate mild anisotropy)")
del _test_weights, _s

In [ ]:
def forward(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

print("forward(): Computes y = W_L @ ... @ W_1 @ X")
print(f"  Input shape:  ({DIM}, {BATCH_SIZE})")
print(f"  Output shape: ({DIM}, {BATCH_SIZE})")

In [ ]:
def compute_loss(weights, X, Y):
    pred = forward(weights, X)
    return 0.5 * np.mean(np.sum((pred - Y)**2, axis=0))

print("compute_loss(): L = 0.5 * mean_over_batch( ||y_pred - y_target||^2 )")
print("  MSE loss, standard regression objective.")

In [ ]:
def compute_gradients(weights, X, Y):
    L = len(weights)
    N = X.shape[1]
    acts = [X.copy()]
    for W in weights:
        acts.append(W @ acts[-1])
    delta = (acts[-1] - Y) / N
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ acts[l].T
        if l > 0:
            delta = weights[l].T @ delta
    return grads

print("compute_gradients(): Standard backpropagation through the linear chain.")
print(f"  Returns {NUM_LAYERS} gradient matrices, each {DIM}x{DIM}.")
print(f"  Each gradient has up to {DIM} non-zero singular values to analyze.")

# Utilization Measurement: The Core Metric

## Per-SV Utilization

Given a gradient G with SVD G = U diag(sigma) V^T, the rank-1 components {u_i v_i^T} form an orthonormal basis for the "directions" that the gradient naturally decomposes into. For any update matrix Delta_W, we can measure how much of it aligns with each direction:

**utilization_i = |u_i^T Delta_W v_i|**

This is the absolute value of the inner product (in the Frobenius sense) between Delta_W and the rank-1 matrix u_i v_i^T. If Delta_W = G itself, then utilization_i = sigma_i (the i-th singular value).

## Utilization Entropy

Normalize to get a probability distribution: p_i = utilization_i / sum_j(utilization_j).

The **Shannon entropy** H = -sum(p_i log2(p_i)) measures how spread out the utilization is:
- **H = 0**: All utilization concentrated in a single SV direction (perfectly anisotropic)
- **H = H_max = log2(32) = 5.0**: Perfectly uniform utilization across all 32 SV directions
- Intermediate values indicate partial concentration

## Why This Metric Distinguishes Muon from Normalized SGD

Both Muon and normalized SGD produce unit-norm updates. But:
- **Normalized SGD**: Delta_W = G / ||G||_F = U diag(sigma / ||sigma||) V^T. The normalization factor cancels from the utilization ratio, so **p_i = sigma_i / sum(sigma_j)** -- identical to raw SGD!
- **Muon**: Delta_W ~ U V^T (polar factor). Then utilization_i ~ |u_i^T (U V^T) v_i| = 1 for all i, giving **p_i = 1/32** -- perfectly uniform!

This is why utilization entropy is the *right* metric to distinguish these optimizers: it is invariant to overall scale but sensitive to the internal distribution of gradient energy across SV directions.

In [ ]:
def compute_utilization(update, U, V):
    """
    Compute per-SV utilization of update matrix.
    Returns array of |<update, u_i v_i^T>| for each i.
    """
    k = min(U.shape[1], V.shape[1])
    util = np.zeros(k)
    for i in range(k):
        # Projection onto rank-1 component u_i v_i^T
        util[i] = abs(np.dot(U[:, i], update @ V[:, i]))
    return util

print("compute_utilization(): Projects update onto each SV direction of the gradient.")
print(f"  For a {DIM}x{DIM} matrix, returns {DIM} utilization values.")
print(f"  utilization_i = |u_i^T @ update @ v_i| -- Frobenius inner product with rank-1 basis element.")

In [ ]:
def entropy(probs):
    """Shannon entropy of a probability distribution."""
    p = probs[probs > 1e-30]
    return -np.sum(p * np.log2(p))

# Verify entropy function with known distributions
_uniform = np.ones(DIM) / DIM
_peaked = np.zeros(DIM); _peaked[0] = 1.0
_two_equal = np.zeros(DIM); _two_equal[0] = 0.5; _two_equal[1] = 0.5
print("Entropy function verification:")
print(f"  Uniform over {DIM} bins:    H = {entropy(_uniform):.4f} bits  (expected: {np.log2(DIM):.4f})")
print(f"  Delta (all mass on 1 bin): H = {entropy(_peaked):.4f} bits  (expected: 0.0000)")
print(f"  Two equal bins:            H = {entropy(_two_equal):.4f} bits  (expected: {np.log2(2):.4f})")
del _uniform, _peaked, _two_equal

# Main Experiment: Training Loop with Utilization Tracking

## Protocol

For each of 5 random seeds, we:
1. Generate a fixed (X, Y) regression problem
2. Train three independent copies of the same network -- one per optimizer (SGD, Muon, Normalized SGD)
3. At every training step, for every layer:
   - Compute gradient G and its SVD: G = U diag(sigma) V^T
   - Compute the optimizer's actual update Delta_W
   - Project Delta_W onto each SV direction: utilization_i = |u_i^T Delta_W v_i|
   - Normalize to a distribution p_i and compute Shannon entropy H
   - Record H and the top-1 fraction p_0

## What We Expect

| Optimizer | Update form | Expected utilization | Expected H | Expected top-1 |
|-----------|------------|---------------------|-----------|----------------|
| SGD | G (with momentum) | ~ sigma_i | Low (~2-3 bits) | High (>0.5) |
| Normalized SGD | G / \|\|G\|\|_F | ~ sigma_i / sum(sigma) | Low (~2-3 bits) | High (>0.5) |
| Muon | U V^T (with momentum) | ~ uniform | High (~4.5-5.0 bits) | Low (~1/32) |

**Critical subtlety with momentum:** Momentum mixes gradient information across steps, so the update at step t is a weighted sum of past gradients. The SVD basis changes at each step, so momentum introduces "cross-talk" between bases from different steps. This could potentially blur the utilization distribution even for SGD. The question is whether Muon's equalization signal is strong enough to dominate the momentum-induced mixing.

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print("=" * 100)
    print("H3a: PER-SV GRADIENT UTILIZATION -- Muon vs Normalized SGD")
    print("=" * 100)
    print(f"Network: {NUM_LAYERS}-layer, {DIM}x{DIM}, {NUM_STEPS} steps, {NUM_SEEDS} seeds")
    print(f"Seeds: {seeds}")
    print(f"Learning rate: 0.01 (fixed across all methods)")
    print(f"H_max = log2({DIM}) = {np.log2(DIM):.4f} bits")
    print()

    # Track utilization entropy for each optimizer
    methods = ['sgd', 'muon', 'norm_sgd']
    all_entropies = {m: [] for m in methods}  # lists of per-step entropies
    all_top1_fracs = {m: [] for m in methods}

    # Also track per-seed summaries for reporting
    per_seed_mean_H = {m: [] for m in methods}
    per_seed_mean_top1 = {m: [] for m in methods}

    for seed_idx, seed in enumerate(seeds):
        rng = np.random.RandomState(seed)
        X = rng.randn(DIM, BATCH_SIZE) * 0.3
        Y = rng.randn(DIM, BATCH_SIZE) * 0.3

        print(f"\n{'─' * 80}")
        print(f"SEED {seed_idx + 1}/{NUM_SEEDS} (seed={seed})")
        print(f"{'─' * 80}")
        print(f"  Data: X ~ N(0, 0.09) shape ({DIM},{BATCH_SIZE}), Y ~ N(0, 0.09) shape ({DIM},{BATCH_SIZE})")

        for method in methods:
            weights = init_weights(seed + 5000)
            mom = [np.zeros((DIM, DIM)) for _ in range(NUM_LAYERS)]
            step_entropies = []
            step_top1 = []

            # Track a few diagnostic snapshots
            snapshot_steps = [0, 49, 99, 249, 499]
            diverged = False

            for step in range(NUM_STEPS):
                loss = compute_loss(weights, X, Y)
                if not np.isfinite(loss) or loss > 1e10:
                    if not diverged:
                        print(f"  [{method:>10}] DIVERGED at step {step}, loss={loss:.4e}")
                        diverged = True
                    break
                grads = compute_gradients(weights, X, Y)

                for i in range(NUM_LAYERS):
                    G = grads[i]
                    U_g, sigma_g, Vt_g = np.linalg.svd(G, full_matrices=False)

                    if method == 'sgd':
                        mom[i] = MOMENTUM * mom[i] + G
                        update = mom[i]
                    elif method == 'muon':
                        ortho_g = newton_schulz(G)
                        mom[i] = MOMENTUM * mom[i] + ortho_g
                        update = mom[i]
                    elif method == 'norm_sgd':
                        mom[i] = MOMENTUM * mom[i] + G
                        v_norm = np.linalg.norm(mom[i], 'fro')
                        update = mom[i] / max(v_norm, 1e-12)

                    util = compute_utilization(update, U_g, Vt_g.T)
                    util_sum = np.sum(util)
                    if util_sum > 1e-30:
                        p = util / util_sum
                        step_entropies.append(entropy(p))
                        step_top1.append(p[0])

                    # Print detailed snapshot at selected steps (layer 0 only for brevity)
                    if step in snapshot_steps and i == 0:
                        if util_sum > 1e-30:
                            p_snap = util / util_sum
                            H_snap = entropy(p_snap)
                            print(f"  [{method:>10}] step={step:>3}, layer=0: "
                                  f"loss={loss:.4e}, H={H_snap:.3f} bits ({H_snap/np.log2(DIM)*100:.1f}% of max), "
                                  f"top1={p_snap[0]:.4f}, top3_frac={np.sum(np.sort(p_snap)[-3:]):.4f}, "
                                  f"grad_cond={sigma_g[0]/max(sigma_g[-1],1e-30):.1f}")

                    weights[i] = weights[i] - 0.01 * update

            # Per-seed summary
            if len(step_entropies) > 0:
                seed_mean_H = np.mean(step_entropies)
                seed_mean_t1 = np.mean(step_top1)
                per_seed_mean_H[method].append(seed_mean_H)
                per_seed_mean_top1[method].append(seed_mean_t1)

            all_entropies[method].extend(step_entropies)
            all_top1_fracs[method].extend(step_top1)

        # Cross-method comparison for this seed
        print(f"\n  Seed {seed} summary:")
        print(f"  {'Method':>15}  {'Mean H':>8}  {'H/H_max':>8}  {'Top-1':>8}")
        for m in methods:
            if per_seed_mean_H[m]:
                print(f"  {m:>15}  {per_seed_mean_H[m][-1]:>8.3f}  "
                      f"{per_seed_mean_H[m][-1]/np.log2(DIM):>8.3f}  "
                      f"{per_seed_mean_top1[m][-1]:>8.4f}")

    # ==================== AGGREGATE RESULTS ====================
    print(f"\n\n{'=' * 100}")
    print("AGGREGATE RESULTS ACROSS ALL SEEDS")
    print(f"{'=' * 100}")

    print(f"\nPer-seed mean entropy (bits):")
    for m in methods:
        vals = per_seed_mean_H[m]
        print(f"  {m:>15}: {['%.3f' % v for v in vals]}")
    print(f"  Spread shows cross-seed consistency (or lack thereof).")

    # Results
    print(f"\n{'=' * 100}")
    print("UTILIZATION ENTROPY (higher = more democratic use of SV directions)")
    print(f"{'=' * 100}")

    H_max = np.log2(DIM)  # maximum entropy for DIM SVs
    print(f"  Maximum possible entropy (uniform over {DIM} SVs): {H_max:.2f} bits")
    print(f"  Number of measurements per method: {len(all_entropies[methods[0]])}")
    print()

    print(f"  {'Method':>15}  {'Mean H':>8}  {'H/H_max':>8}  {'Top-1 frac':>12}  {'Std H':>8}")
    print("  " + "-" * 55)

    for m in methods:
        h = np.array(all_entropies[m])
        t1 = np.array(all_top1_fracs[m])
        print(f"  {m:>15}  {np.mean(h):>8.3f}  {np.mean(h)/H_max:>8.3f}  "
              f"{np.mean(t1):>12.4f}  {np.std(h):>8.3f}")

    # Additional analysis: distribution percentiles
    print(f"\n  Entropy distribution percentiles:")
    print(f"  {'Method':>15}  {'5th':>8}  {'25th':>8}  {'Median':>8}  {'75th':>8}  {'95th':>8}")
    print("  " + "-" * 55)
    for m in methods:
        h = np.array(all_entropies[m])
        pcts = np.percentile(h, [5, 25, 50, 75, 95])
        print(f"  {m:>15}  {pcts[0]:>8.3f}  {pcts[1]:>8.3f}  {pcts[2]:>8.3f}  {pcts[3]:>8.3f}  {pcts[4]:>8.3f}")

    print(f"\n  INTERPRETATION:")
    print(f"  - If Muon's 5th percentile exceeds SGD's 95th percentile,")
    print(f"    the distributions are completely separated (strongest evidence).")
    print(f"  - If Muon and Norm SGD have similar entropies, the polar factor")
    print(f"    does NOT provide additional democratization beyond scaling.")

    # Hypothesis tests
    print(f"\n\n{'=' * 100}")
    print("HYPOTHESIS TESTS")
    print(f"{'=' * 100}")

    muon_H = np.mean(all_entropies['muon'])
    norm_H = np.mean(all_entropies['norm_sgd'])
    sgd_H = np.mean(all_entropies['sgd'])
    sgd_top1 = np.mean(all_top1_fracs['sgd'])

    t1 = muon_H > norm_H + 0.5
    t2 = muon_H > 0.9 * H_max
    t3 = sgd_top1 > 0.5

    print(f"\n  T1: Muon entropy > Norm SGD entropy + 0.5 bits?")
    print(f"      Muon={muon_H:.3f}, NormSGD={norm_H:.3f}, diff={muon_H-norm_H:.3f}")
    print(f"      Required gap: 0.5 bits. Actual gap: {muon_H-norm_H:.3f} bits.")
    if t1:
        print(f"      Muon extracts {muon_H-norm_H:.3f} more bits of directional diversity.")
        print(f"      This confirms the polar factor is NOT just rescaling -- it redistributes.")
    else:
        print(f"      Gap is less than 0.5 bits. Possible explanations:")
        print(f"        - Momentum mixing blurs the distinction between methods")
        print(f"        - Gradient SVD structure changes between steps (non-stationarity)")
        print(f"        - The utilization is measured w.r.t. current-step SVD basis,")
        print(f"          but momentum accumulates over past steps with different bases")
    print(f"      --> {'PASS' if t1 else 'FAIL'}")

    print(f"\n  T2: Muon utilization nearly uniform (H > 0.9 * H_max)?")
    print(f"      Muon H={muon_H:.3f}, 0.9*H_max={0.9*H_max:.3f}")
    print(f"      Ratio: {muon_H/H_max:.4f} (need > 0.9)")
    if t2:
        print(f"      Muon achieves {muon_H/H_max*100:.1f}% of maximum entropy.")
        print(f"      The optimizer uses information from essentially ALL SV directions.")
    else:
        print(f"      Muon only reaches {muon_H/H_max*100:.1f}% of maximum entropy.")
        print(f"      Momentum or NS approximation may prevent perfect uniformity.")
    print(f"      --> {'PASS' if t2 else 'FAIL'}")

    print(f"\n  T3: SGD top-1 SV fraction > 0.5?")
    print(f"      SGD top-1 = {sgd_top1:.4f}")
    if t3:
        print(f"      Over {sgd_top1*100:.1f}% of SGD's update energy goes to the top SV direction.")
        print(f"      This confirms SGD is dominated by the largest gradient singular value.")
        print(f"      The remaining {(1-sgd_top1)*100:.1f}% is spread across {DIM-1} other directions.")
    else:
        print(f"      SGD top-1 fraction is only {sgd_top1*100:.1f}%.")
        print(f"      Momentum may be distributing energy more evenly than expected.")
    print(f"      --> {'PASS' if t3 else 'FAIL'}")

    # Cross-method comparison
    print(f"\n\n{'=' * 100}")
    print("CROSS-METHOD ANALYSIS")
    print(f"{'=' * 100}")
    print(f"\n  SGD vs Normalized SGD entropy gap:  {norm_H - sgd_H:.3f} bits")
    print(f"    (Expected: ~0 bits, since normalization preserves SV ratios)")
    print(f"    (If large: momentum interaction with norm creates unexpected effects)")
    print(f"\n  Muon vs SGD entropy gap:            {muon_H - sgd_H:.3f} bits")
    print(f"    (This is the 'spectral democracy dividend' of the polar factor)")
    print(f"\n  Muon vs Normalized SGD entropy gap:  {muon_H - norm_H:.3f} bits")
    print(f"    (The pure effect of equalization vs. rescaling)")

    muon_top1 = np.mean(all_top1_fracs['muon'])
    norm_top1 = np.mean(all_top1_fracs['norm_sgd'])
    print(f"\n  Top-1 SV fractions:")
    print(f"    SGD:          {sgd_top1:.4f}  (expected: high, ~0.3-0.5+)")
    print(f"    Normalized SGD: {norm_top1:.4f}  (expected: similar to SGD)")
    print(f"    Muon:         {muon_top1:.4f}  (expected: ~{1.0/DIM:.4f} = 1/{DIM} for uniform)")

    print(f"\n{'=' * 100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'=' * 100}")

In [ ]:
if __name__ == '__main__':
    main()

# Conclusions and Interpretation

## What This Experiment Reveals

This experiment directly measures the **spectral democracy** property of Muon vs. other optimizers. The key findings should be interpreted as follows:

### If T1 PASSES (Muon entropy > Norm SGD entropy + 0.5 bits):
The polar factor provides a qualitatively different kind of normalization than simple Frobenius rescaling. While normalized SGD preserves the relative weighting of SV directions (just rescaling the magnitude), Muon actively **equalizes** them. This means Muon extracts useful update information from gradient directions that SGD and normalized SGD effectively ignore.

### If T2 PASSES (Muon H > 0.9 * H_max):
Muon achieves near-uniform utilization, confirming the theoretical prediction that the polar factor maps all singular values to 1. Even with momentum (which mixes gradients from different steps with different SVD bases), the equalization signal dominates.

### If T3 PASSES (SGD top-1 fraction > 0.5):
SGD is heavily dominated by the top singular value of the gradient, meaning most of the update's effect is in a single direction. This creates an effective "rank-1 bottleneck" where 31 out of 32 gradient directions contribute minimally.

## Connection to RG Gauge-Fixing Interpretation

In the renormalization group view, Muon's equalization of SV utilization corresponds to **fixing the gauge** so that all energy scales contribute equally to the flow. SGD's concentration on the top SV is analogous to a gauge where only the UV (high-energy) mode drives the RG flow -- all IR modes are suppressed. The polar factor acts as a gauge transformation that democratizes the multi-scale structure of the gradient.

## Caveats and Limitations

1. **Momentum cross-talk**: We measure utilization w.r.t. the *current* step's gradient SVD basis, but momentum accumulates updates from *past* steps with different bases. This introduces off-diagonal terms that could blur the distinction.
2. **Linear network**: Nonlinearities would introduce additional structure in the gradient SVD that might affect utilization patterns differently.
3. **Fixed learning rate**: All methods use lr=0.01. Different optimal learning rates could change the effective utilization through nonlinear dynamics.
4. **Small scale**: 32x32 matrices have only 32 SV directions. In real networks with thousands of SVs, the entropy gap could be much larger.